In [1]:
import matplotlib.pyplot as plt
import numpy as np
from synthetic_data import distributions, generate_ground_truths, generate_sample_csv

In [2]:
def owens_bozic_model(y, t, params):
    T, E, C, M = y
    a, b, dE, dC, g, jE, jC, K, k, l, mE, mC, qE, qC, s, KT, KE, KC, gamma = params

    C = max(C, 0)
    E = max(E, 0)

    tol = 1e-10

    if T < tol:
        DE = 0
    elif E < T:
        DE = dE * (E / T)**l / (s + (E / T)**l) * T
    else:
        DE = dE * (1 - s / (s + (T/E)**(-l))) * T

    if T < tol:
        DC = 0
    elif C < T:
        DC = dC * (C / T)**l / (s + (C / T)**l) * T
    else:
        DC = dC * (1 - s / (s + (T/C)**(-l))) * T

    dT_dt = a * T * (1 - b * T) - DE - DC - KT * (1 - np.exp(-M)) * T
    dE_dt = g - mE * E - jE * np.log((E + C) / K) * (DE**2) / (k + DE**2) * E  - qE * E * T - KE * (1 - np.exp(-M)) * E
    dC_dt = - mC * C - jC * np.log((E + C) / K) * (DC**2) / (k + DC**2) * C - qC * C * T - KC * (1 - np.exp(-M)) * C
    dM_dt = - gamma * M

    return [dT_dt, dE_dt, dC_dt, dM_dt]

In [ ]:
def csv_and_indiv_param_vals(
    param: str,
    noise_level: float,
    n_obs: int,
    param_seed: int,
    noise_seed: int,
    csv_filename: str | None = None,
    n_indivs=10,
    max_day_number=100,
):
    f_pop = 1.0  # scale factor for pop param std

    partial_pop_param_info = [
        ("a", 3.3e-1, 0.6 * f_pop, distributions["lognormal"]),
        ("b", 2.26e-11, 5.1 * f_pop, distributions["lognormal"]),
        ("dE", 3.17, 0.6 * f_pop, distributions["lognormal"]),
        ("dC", 2.25, 0.01 * f_pop, distributions["lognormal"]),
        ("g", 1.03e4, 1.7 * f_pop, distributions["lognormal"]),
        ("jE", 1.56e-2, 0.75 * f_pop, distributions["lognormal"]),
        ("jC", 3.46e-1, 0.75 * f_pop, distributions["lognormal"]),
        ("K", 5.21e8, 1.5 * f_pop, distributions["lognormal"]),
        ("k", 8.67e6, 3.0 * f_pop, distributions["lognormal"]),
        ("l", 1.418, 0.013 * f_pop, distributions["lognormal"]),
        ("mE", 1.76e-2, 0.76 * f_pop, distributions["lognormal"]),
        ("mC", 0.293, 0.01 * f_pop, distributions["lognormal"]),
        ("qE", 1.28e-10, 1.35 * f_pop, distributions["lognormal"]),
        ("qC", 2.14e-10, 2.6 * f_pop, distributions["lognormal"]),
        ("s", 3.02e-1, 0.17 * f_pop, distributions["lognormal"]),
        ("KT", 0.7, 0.01 * f_pop, distributions["lognormal"]),
        ("KE", 0.6, 0.01 * f_pop, distributions["lognormal"]),
        ("KC", 0.6, 0.01 * f_pop, distributions["lognormal"]),
        ("gamma", 0.9, 0.01 * f_pop, distributions["lognormal"]),
    ]

    param_idx = -1
    for i, x in enumerate(partial_pop_param_info):
        if x[0] == param:
            param_idx = i
            break

    if param_idx == -1:
        raise ValueError(f"unknown param {param}")

    pop_param_info = [(*x, x[0] != param) for x in partial_pop_param_info]
    obs_var_info = [
        (
            "T",
            1.58e9,
            1.5,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "E",
            4e5,
            0.01,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "C",
            6.3e7,
            2.0,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "M",
            6.0,
            0.2,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
    ]
    obs_times_all = [
        [max_day_number * i // n_obs for i in range(1, n_obs + 1)]
        for _ in range(n_indivs)
    ]
    sampled_params_all, initial_conditions_all, _, _, _ = generate_sample_csv(
        csv_filename or "owens_bozic_noiseexp",
        owens_bozic_model,
        pop_param_info,
        obs_var_info,
        n_indivs,
        n_obs,
        max_day_number,
        obs_times_all=obs_times_all,
        param_seed=param_seed,
        noise_seed=noise_seed,
    )

    return (
        [x[param_idx] for x in sampled_params_all],
        sampled_params_all,
        initial_conditions_all,
        param_idx,
    )


def csv_and_param_subset_vals(
    params: list[str],
    noise_level: float,
    n_obs: int,
    param_seed: int,
    noise_seed: int,
    csv_filename: str | None = None,
    n_indivs=10,
    max_day_number=100,
):
    f_pop = 1.0  # scale factor for pop param std

    partial_pop_param_info = [
        ("a", 3.3e-1, 0.6 * f_pop, distributions["lognormal"]),
        ("b", 2.26e-11, 5.1 * f_pop, distributions["lognormal"]),
        ("dE", 3.17, 0.6 * f_pop, distributions["lognormal"]),
        ("dC", 2.25, 0.01 * f_pop, distributions["lognormal"]),
        ("g", 1.03e4, 1.7 * f_pop, distributions["lognormal"]),
        ("jE", 1.56e-2, 0.75 * f_pop, distributions["lognormal"]),
        ("jC", 3.46e-1, 0.75 * f_pop, distributions["lognormal"]),
        ("K", 5.21e8, 1.5 * f_pop, distributions["lognormal"]),
        ("k", 8.67e6, 3.0 * f_pop, distributions["lognormal"]),
        ("l", 1.418, 0.013 * f_pop, distributions["lognormal"]),
        ("mE", 1.76e-2, 0.76 * f_pop, distributions["lognormal"]),
        ("mC", 0.293, 0.01 * f_pop, distributions["lognormal"]),
        ("qE", 1.28e-10, 1.35 * f_pop, distributions["lognormal"]),
        ("qC", 2.14e-10, 2.6 * f_pop, distributions["lognormal"]),
        ("s", 3.02e-1, 0.17 * f_pop, distributions["lognormal"]),
        ("KT", 0.7, 0.01 * f_pop, distributions["lognormal"]),
        ("KE", 0.6, 0.01 * f_pop, distributions["lognormal"]),
        ("KC", 0.6, 0.01 * f_pop, distributions["lognormal"]),
        ("gamma", 0.9, 0.01 * f_pop, distributions["lognormal"]),
    ]

    param_idxs = [-1, -1]
    for j, param in enumerate(params):
        for i, x in enumerate(partial_pop_param_info):
            if x[0] == param:
                param_idxs[j] = i
                break

        if param_idxs[j] == -1:
            raise ValueError(f"unknown param {param}")
    print(param_idxs)

    pop_param_info = [(*x, x[0] not in params) for x in partial_pop_param_info]
    obs_var_info = [
        (
            "T",
            1.58e9,
            1.5,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "E",
            4e5,
            0.01,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "C",
            6.3e7,
            2.0,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
        (
            "M",
            6.0,
            0.2,
            distributions["normal"],
            "combined1",
            0,
            noise_level,
            1,
            distributions["normal"],
            "log10",
            True,
        ),
    ]
    obs_times_all = [
        [max_day_number * i // n_obs for i in range(1, n_obs + 1)]
        for _ in range(n_indivs)
    ]
    sampled_params_all, initial_conditions_all, _, _, _ = generate_sample_csv(
        csv_filename or "owens_bozic_noiseexp",
        owens_bozic_model,
        pop_param_info,
        obs_var_info,
        n_indivs,
        n_obs,
        max_day_number,
        obs_times_all=obs_times_all,
        param_seed=param_seed,
        noise_seed=noise_seed,
    )

    return (
        [[x[param_idx] for param_idx in param_idxs] for x in sampled_params_all],
        sampled_params_all,
        initial_conditions_all,
        param_idxs,
    )


In [4]:
def get_info_for_ground_truths(
    param: str, noise_level: float, param_seed: int, noise_seed: int
):
    _, sampled_params_all, initial_conditions_all, param_idxs = csv_and_indiv_param_vals(
        param, noise_level, 1, param_seed, noise_seed, n_indivs=1
    )
    return sampled_params_all, initial_conditions_all, param_idxs


In [ ]:
sampled_params_all, initial_conditions_all, param_idxs = get_info_for_ground_truths(
    "a", 0.25, 9, 69420
)
ground_truths = generate_ground_truths(
    owens_bozic_model, sampled_params_all, initial_conditions_all
)

[[0.18766125904922945, 2.2599999999999963e-11, 3.17, 2.25, 10300.000000000005, 0.015600000000000001, 0.34600000000000003, 520999999.9999993, 8670000.000000004, 1.418, 0.017600000000000008, 0.293, 1.2799999999999997e-10, 2.1399999999999996e-10, 0.30200000000000005, 0.7, 0.6, 0.6, 0.9]] [[1580000000.0, 400000.0, 63000000.0, 6.0]] 0
